In [22]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, f_classif

In [23]:
df = pd.DataFrame({
    'EmployeeID': [101,102,103,104,105,106,107,108],
    'Department': ['Sales','R&D','HR','Sales','R&D','HR','Sales','R&D'],
    'JoinDate': pd.to_datetime([
        '2018-03-15',
        '2020-07-10',
        '2019-01-25',
        '2021-11-05',
        '2017-06-20',
        '2022-02-14',
        '2019-09-30',
        '2020-12-01'
    ]),
    'MonthlySalary': [5000,7000,4500,6500,8000,4800,5500,7200],
    'YearsAtCompany': [5,3,4,2,6,1,4,3],
    'NumProjects': [4,6,3,5,7,2,4,5],
    'LastReviewScore': [4.2,4.8,3.9,4.5,4.9,3.7,4.1,4.6],
    'Attrition': [0,1,0,1,0,1,0,1]
})

print(df)

   EmployeeID Department   JoinDate  MonthlySalary  YearsAtCompany  \
0         101      Sales 2018-03-15           5000               5   
1         102        R&D 2020-07-10           7000               3   
2         103         HR 2019-01-25           4500               4   
3         104      Sales 2021-11-05           6500               2   
4         105        R&D 2017-06-20           8000               6   
5         106         HR 2022-02-14           4800               1   
6         107      Sales 2019-09-30           5500               4   
7         108        R&D 2020-12-01           7200               3   

   NumProjects  LastReviewScore  Attrition  
0            4              4.2          0  
1            6              4.8          1  
2            3              3.9          0  
3            5              4.5          1  
4            7              4.9          0  
5            2              3.7          1  
6            4              4.1          0  
7        

In [24]:
df['Workload_Index'] = (df['YearsAtCompany']*df['NumProjects'])
df['Salary_per_Project'] = (df['MonthlySalary']/df['NumProjects'])
reference_date = df['JoinDate'].max()
df['Days_Since_Join'] = (reference_date-df['JoinDate']).dt.days
print(df[['Workload_Index',
          'Salary_per_Project',
          'Days_Since_Join']].head())

   Workload_Index  Salary_per_Project  Days_Since_Join
0              20         1250.000000             1432
1              18         1166.666667              584
2              12         1500.000000             1116
3              10         1300.000000              101
4              42         1142.857143             1700


In [25]:
df['Join_Month'] = df['JoinDate'].dt.month
print(df['Join_Month'])
df['Month_sin'] = np.sin(2*np.pi*df['Join_Month']/12)
df['Month_cos'] = np.cos(2*np.pi*df['Join_Month']/12)

0     3
1     7
2     1
3    11
4     6
5     2
6     9
7    12
Name: Join_Month, dtype: int32


In [26]:
df=pd.get_dummies(df,columns=['Department'],drop_first=True)
print(df.columns)

Index(['EmployeeID', 'JoinDate', 'MonthlySalary', 'YearsAtCompany',
       'NumProjects', 'LastReviewScore', 'Attrition', 'Workload_Index',
       'Salary_per_Project', 'Days_Since_Join', 'Join_Month', 'Month_sin',
       'Month_cos', 'Department_R&D', 'Department_Sales'],
      dtype='object')


In [27]:
df = df.drop(
    columns=[
        'EmployeeID',
        'JoinDate',
        'Join_Month'
    ]
)

X = df.drop(columns=['Attrition'])
y = df['Attrition']
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)
selector = VarianceThreshold(
    threshold=0.01
)
selector.fit(X_scaled)
kept = X.columns[
    selector.get_support()
].tolist()
print("Kept columns:")
print(kept)
X_vt = selector.transform(X_scaled)
X_vt = pd.DataFrame(
    X_vt,
    columns=kept
)

Kept columns:
['MonthlySalary', 'YearsAtCompany', 'NumProjects', 'LastReviewScore', 'Workload_Index', 'Salary_per_Project', 'Days_Since_Join', 'Month_sin', 'Month_cos', 'Department_R&D', 'Department_Sales']


In [28]:
selector_k = SelectKBest(
    score_func=f_classif,
    k=6
)
selector_k.fit(X_vt, y)
scores = pd.Series(
    selector_k.scores_,
    index=X_vt.columns
)

print(scores.sort_values(
    ascending=False
))
final_cols = X_vt.columns[
    selector_k.get_support()
].tolist()

print("Final six columns:")
print(final_cols)

Days_Since_Join       1.912622e+01
YearsAtCompany        1.363636e+01
Workload_Index        2.214728e+00
Salary_per_Project    7.990618e-01
Month_cos             5.091275e-01
MonthlySalary         4.333256e-01
Department_Sales      4.285714e-01
Department_R&D        4.285714e-01
LastReviewScore       1.479290e-01
Month_sin             8.780709e-02
NumProjects           1.040002e-32
dtype: float64
Final six columns:
['MonthlySalary', 'YearsAtCompany', 'Workload_Index', 'Salary_per_Project', 'Days_Since_Join', 'Month_cos']


In [29]:
final_df = X_vt[
    final_cols
].copy()

final_df['Attrition'] = y
final_df.to_csv(
    'mini_project.csv',
    index=False
)